## ライブラリ・データの読み込み

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier

In [3]:
ROOT = Path.cwd()

if ROOT.name == "smartphone_addiction":
    ROOT = ROOT.parent.parent

DATA_DIR = ROOT / "data" / "smartphone_addiction" / "raw"

train = pd.read_csv(DATA_DIR / "train.csv")

TARGET = "addicted_label"

X = train.drop(
    columns=[TARGET, "id"]
).copy()

y = train[TARGET].copy()

In [4]:
# カテゴリ列
categorical_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in categorical_cols:
    X[col] = (
        X[col]
        .fillna("Missing")
        .astype(str)
    )

print(categorical_cols)

['gender', 'stress_level', 'academic_work_impact']


C:\Users\Owner\AppData\Local\Temp\ipykernel_25156\2100417684.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


## 1-fold 高速比較用の関数を作成

In [5]:
# fold1を固定
# 全候補を同じ validation で評価できる
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

train_idx, valid_idx = next(
    cv.split(X, y)
)

X_train = X.iloc[train_idx]
X_valid = X.iloc[valid_idx]

y_train = y.iloc[train_idx]
y_valid = y.iloc[valid_idx]

print(X_train.shape)
print(X_valid.shape)

(460912, 12)
(230457, 12)


In [6]:
# 1-fold 評価関数
def evaluate_one_fold(params):
    model = CatBoostClassifier(
        iterations=7000,
        learning_rate=0.05,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        **params
    )

    model.fit(
        X_train,
        y_train,
        cat_features=categorical_cols,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=200
    )

    pred = model.predict_proba(
        X_valid
    )[:, 1]

    auc = roc_auc_score(
        y_valid,
        pred
    )

    return {
        **params,
        "auc": auc,
        "best_iteration":
            model.get_best_iteration()
    }

## depth：tree の複雑さ
- depth=7 なら、１本の tree で最大７段程度の条件を組み合わせられる
- 深くすると、複雑な interaction を表現可能
- 一方で、過学習しやすい・計算量が増える　といったトレードオフな関係でもある

In [7]:
depth_results = []

for depth in [5, 6, 7, 8, 9]:
    print("depth =", depth)

    result = evaluate_one_fold({
        "depth": depth
    })

    depth_results.append(result)

depth_df = pd.DataFrame(
    depth_results
).sort_values(
    "auc",
    ascending=False
)

display(depth_df)

depth = 5
depth = 6
depth = 7
depth = 8
depth = 9


,depth,auc,best_iteration
1,6,0.963184,6967
2,7,0.963044,5793
0,5,0.963009,6989
3,8,0.962772,4155
4,9,0.962498,3833


In [8]:
# Best Depth を固定
BEST_DEPTH = int(
    depth_df.iloc[0]["depth"]
)

print(BEST_DEPTH)

6


## L2 reguration：モデルをどれくらい正則化するか
- 小さい：自由に学習するため、過学習リスクが上がる
- 大きい：慎重に学習するため、過学習リスクは低いが、大きすぎると学習不足になる可能性もある

In [9]:
l2_results = []

for l2 in [1, 3, 5, 10, 20]:
    print("l2_leaf_reg =", l2)

    result = evaluate_one_fold({
        "depth": BEST_DEPTH,
        "l2_leaf_reg": l2
    })

    l2_results.append(result)

l2_df = pd.DataFrame(
    l2_results
).sort_values(
    "auc",
    ascending=False
)

display(l2_df)

l2_leaf_reg = 1
l2_leaf_reg = 3
l2_leaf_reg = 5
l2_leaf_reg = 10
l2_leaf_reg = 20


,depth,l2_leaf_reg,auc,best_iteration
2,6,5,0.963205,6937
1,6,3,0.963184,6967
3,6,10,0.963176,6999
0,6,1,0.963158,6994
4,6,20,0.963157,6997


In [10]:
BEST_L2 = float(
    l2_df.iloc[0]["l2_leaf_reg"]
)

## random_strength：Tree の分岐を決めるときに少しランダム性を持たせるパラメータ
- 適度なランダム性であることで、特定のデータにぴったりと合う Tree の作成を防ぐ

In [11]:
random_results = []

for rs in [0, 0.5, 1, 2, 5]:
    print("random_strength =", rs)

    result = evaluate_one_fold({
        "depth": BEST_DEPTH,
        "l2_leaf_reg": BEST_L2,
        "random_strength": rs
    })

    random_results.append(result)

random_df = pd.DataFrame(
    random_results
).sort_values(
    "auc",
    ascending=False
)

display(random_df)

random_strength = 0
random_strength = 0.5
random_strength = 1
random_strength = 2
random_strength = 5


,depth,l2_leaf_reg,random_strength,auc,best_iteration
1,6,5.0,0.5,0.963209,6994
2,6,5.0,1.0,0.963205,6937
0,6,5.0,0.0,0.963181,6973
3,6,5.0,2.0,0.963165,6754
4,6,5.0,5.0,0.963136,6998


In [12]:
BEST_RANDOM = float(
    random_df.iloc[0]["random_strength"]
)

## bagging_temparature
- 過学習を抑えたり、モデルに多様性を持たせたりする

In [13]:
bagging_results = []

for bt in [0, 0.5, 1, 2, 5]:
    print("bagging_temperature =", bt)

    result = evaluate_one_fold({
        "depth": BEST_DEPTH,
        "l2_leaf_reg": BEST_L2,
        "random_strength": BEST_RANDOM,
        "bagging_temperature": bt
    })

    bagging_results.append(result)

bagging_df = pd.DataFrame(
    bagging_results
).sort_values(
    "auc",
    ascending=False
)

display(bagging_df)

bagging_temperature = 0
bagging_temperature = 0.5
bagging_temperature = 1
bagging_temperature = 2
bagging_temperature = 5


,depth,l2_leaf_reg,random_strength,bagging_temperature,auc,best_iteration
0,6,5.0,0.5,0.0,0.963209,6994
1,6,5.0,0.5,0.5,0.963209,6994
2,6,5.0,0.5,1.0,0.963209,6994
3,6,5.0,0.5,2.0,0.963209,6994
4,6,5.0,0.5,5.0,0.963209,6994


## Best parameter をまとめる

In [14]:
BEST_BAGGING = float(
    bagging_df.iloc[0][
        "bagging_temperature"
    ]
)

best_params = {
    "depth": BEST_DEPTH,
    "l2_leaf_reg": BEST_L2,
    "random_strength": BEST_RANDOM,
    "bagging_temperature": BEST_BAGGING
}

print(best_params)

{'depth': 6, 'l2_leaf_reg': 5.0, 'random_strength': 0.5, 'bagging_temperature': 0.0}


In [15]:
# たまたま 1-fold にはまった可能性があるため、最終候補は 3-fold 行う
def evaluate_three_folds(params):
    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    scores = []
    best_iterations = []

    for fold, (
        train_idx,
        valid_idx
    ) in enumerate(
        cv.split(X, y),
        start=1
    ):
        print(
            f"\n===== Fold {fold} ====="
        )

        X_train_fold = X.iloc[
            train_idx
        ]

        X_valid_fold = X.iloc[
            valid_idx
        ]

        y_train_fold = y.iloc[
            train_idx
        ]

        y_valid_fold = y.iloc[
            valid_idx
        ]

        model = CatBoostClassifier(
            iterations=7000,
            learning_rate=0.05,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=42,
            verbose=500,
            allow_writing_files=False,
            **params
        )

        model.fit(
            X_train_fold,
            y_train_fold,
            cat_features=
                categorical_cols,
            eval_set=(
                X_valid_fold,
                y_valid_fold
            ),
            early_stopping_rounds=200
        )

        pred = model.predict_proba(
            X_valid_fold
        )[:, 1]

        score = roc_auc_score(
            y_valid_fold,
            pred
        )

        scores.append(score)

        best_iterations.append(
            model.get_best_iteration()
        )

        print(
            f"AUC: {score:.6f}"
        )

    return {
        "mean_auc":
            np.mean(scores),

        "std_auc":
            np.std(scores),

        "scores":
            scores,

        "best_iterations":
            best_iterations
    }

In [17]:
# 実行
tuned_result = evaluate_three_folds(
    best_params
)

tuned_result


===== Fold 1 =====
0:	test: 0.9055699	best: 0.9055699 (0)	total: 246ms	remaining: 28m 42s
500:	test: 0.9509404	best: 0.9509404 (500)	total: 1m 39s	remaining: 21m 30s
1000:	test: 0.9566083	best: 0.9566083 (1000)	total: 3m 15s	remaining: 19m 31s
1500:	test: 0.9594375	best: 0.9594375 (1500)	total: 4m 54s	remaining: 18m
2000:	test: 0.9608339	best: 0.9608339 (2000)	total: 6m 32s	remaining: 16m 21s
2500:	test: 0.9615864	best: 0.9615864 (2500)	total: 8m 9s	remaining: 14m 41s
3000:	test: 0.9620479	best: 0.9620479 (3000)	total: 9m 47s	remaining: 13m 2s
3500:	test: 0.9624016	best: 0.9624016 (3500)	total: 11m 25s	remaining: 11m 24s
4000:	test: 0.9626159	best: 0.9626160 (3999)	total: 13m 3s	remaining: 9m 46s
4500:	test: 0.9627749	best: 0.9627752 (4496)	total: 14m 42s	remaining: 8m 9s
5000:	test: 0.9629104	best: 0.9629120 (4991)	total: 16m 20s	remaining: 6m 31s
5500:	test: 0.9629992	best: 0.9629995 (5499)	total: 17m 58s	remaining: 4m 53s
6000:	test: 0.9630862	best: 0.9630869 (5995)	total: 19m 36s	

{'mean_auc': np.float64(0.9636866861540904),
 'std_auc': np.float64(0.0003378248528503061),
 'scores': [0.9632090037748509, 0.9639328146771102, 0.9639182400103099],
 'best_iterations': [6994, 6998, 6844]}

## 結果を比較

In [18]:
comparison = pd.DataFrame([
    {
        "model":
            "CatBoost baseline",
        "mean_auc":
            0.963516,
        "std_auc":
            0.000343
    },
    {
        "model":
            "CatBoost tuned",
        "mean_auc":
            tuned_result[
                "mean_auc"
            ],
        "std_auc":
            tuned_result[
                "std_auc"
            ]
    }
])

display(comparison)

,model,mean_auc,std_auc
0,CatBoost baseline,0.963516,0.000343
1,CatBoost tuned,0.963687,0.000338


## learning_rate も比較

In [19]:
def evaluate_learning_rate(lr):
    model = CatBoostClassifier(
        iterations=10000,
        learning_rate=lr,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        **best_params
    )

    model.fit(
        X_train,
        y_train,
        cat_features=categorical_cols,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=300
    )

    pred = model.predict_proba(
        X_valid
    )[:, 1]

    return {
        "learning_rate": lr,
        "auc":
            roc_auc_score(
                y_valid,
                pred
            ),
        "best_iteration":
            model.get_best_iteration()
    }

In [ ]:
lr_results = []

for lr in [
    0.03,
    0.05,
    0.07,
    0.1
]:
    print(
        "learning_rate =",
        lr
    )

    lr_results.append(
        evaluate_learning_rate(lr)
    )

lr_df = pd.DataFrame(
    lr_results
).sort_values(
    "auc",
    ascending=False
)

display(lr_df)

learning_rate = 0.03
learning_rate = 0.05
